# SOXL 目标可行性与可预测性审计 v1

先判断联合风险/收益门槛是否存在可行域，再判断现有特征能否稳定预测需要的标签。2011–2022 为开发期；2023–2026 outer 只诊断，绝不参与选择。

In [1]:
from pathlib import Path
import json
import sys
repo = Path.cwd().resolve()
if repo.name == 'soxl_objective_feasibility_and_predictability_audit_v1':
    repo = repo.parents[1]
sys.path.insert(0, repo.as_posix())
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.workflow.soxl_objective_feasibility_audit import run_objective_feasibility_audit_card

provider = repo / 'data' / 'qlib' / 'soxl_soxx_soxs'
qlib.init(provider_uri=provider.as_posix(), region='us', kernels=1)
calendar = D.calendar(start_time='2011-01-03', end_time='2026-07-21', freq='day')
summary = run_objective_feasibility_audit_card(root=repo)
recorder = R.get_recorder(recorder_id=summary['recorder_id'], experiment_name=summary['experiment'])
{'calendar_rows': len(calendar), 'recorder_id': recorder.id, 'decision': summary['promotion_decision']}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[9811:MainThread](2026-07-22 20:33:16,778) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[9811:MainThread](2026-07-22 20:33:16,780) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[9811:MainThread](2026-07-22 20:33:16,781) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs')}


[9811:MainThread](2026-07-22 20:33:16,787) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[9811:MainThread](2026-07-22 20:33:16,819) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[9811:MainThread](2026-07-22 20:33:24,329) INFO - qlib.workflow - [exp.py:258] - Experiment 761868714399348284 starts running ...


[9811:MainThread](2026-07-22 20:33:24,420) INFO - qlib.workflow - [recorder.py:345] - Recorder a2c94b3e1953489ca281dc3160d4a7dc starts running under Experiment 761868714399348284 ...


[9811:MainThread](2026-07-22 20:33:24,891) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'calendar_rows': 3909,
 'recorder_id': 'a2c94b3e1953489ca281dc3160d4a7dc',
 'decision': 'needs_iteration_feasible_oracle_and_some_stable_targets_but_no_tradable_rule_pass'}

In [2]:
card = repo / 'research' / summary['experiment']
gates = pd.read_csv(card / 'feasibility_gate_summary.csv')
display(gates)
frontier = pd.read_csv(card / 'feasibility_frontier.csv')
display(frontier.query("segment == 'development' and cost_bps == 10").sort_values(['all_primary_gates_pass', 'return_maxdd'], ascending=False)[['strategy', 'strategy_kind', 'return_pct', 'terminal_wealth_ratio_vs_soxx', 'max_drawdown_pct', 'worst_rolling_5d_pct', 'upside_capture_pct', 'all_primary_gates_pass']].head(15))

,segment,strategy_kind,cost_bps,strategies,primary_gate_passes,risk_gate_passes,position_gate_passes,capture_gate_passes,relative_soxx_gate_passes,efficiency_gate_passes
0,development,mechanical_tradable,10,19,0,2,18,0,2,1
1,development,mechanical_tradable,25,19,0,2,18,0,2,1
2,development,oracle_nontradable,10,10,2,8,10,4,10,10
3,development,oracle_nontradable,25,10,2,8,10,4,10,10
4,outer,mechanical_tradable,10,19,0,2,16,1,2,0
5,outer,mechanical_tradable,25,19,0,2,16,1,2,0
6,outer,oracle_nontradable,10,10,2,8,10,4,10,10
7,outer,oracle_nontradable,25,10,2,8,10,4,10,10


,strategy,strategy_kind,return_pct,terminal_wealth_ratio_vs_soxx,max_drawdown_pct,worst_rolling_5d_pct,upside_capture_pct,all_primary_gates_pass
44,oracle_with_soxs_lambda_10,oracle_nontradable,2.096376e+17,2.916205e+14,-9.564161,-8.155722,66.265132,True
42,oracle_long_only_lambda_10,oracle_nontradable,1.207547e+16,1.679782e+13,-9.564161,-8.155722,68.040671,True
40,oracle_with_soxs_lambda_0,oracle_nontradable,8.691695e+17,1.209075e+15,-20.609975,-15.688454,76.763632,False
38,oracle_long_only_lambda_0,oracle_nontradable,5.013334e+16,6.973895e+13,-20.609975,-16.872494,78.639990,False
48,oracle_with_soxs_lambda_50,oracle_nontradable,4.432236e+15,6.165549e+12,-5.009765,-5.009765,53.112292,False
46,oracle_long_only_lambda_50,oracle_nontradable,2.729524e+14,3.796958e+11,-5.009765,-5.009765,54.561363,False
52,oracle_with_soxs_lambda_200,oracle_nontradable,6.595598e+13,9.174935e+10,-2.121820,-1.920479,43.349024,False
50,oracle_long_only_lambda_200,oracle_nontradable,5.115080e+12,7.115432e+09,-2.121820,-1.887516,44.231110,False
56,oracle_with_soxs_lambda_1000,oracle_nontradable,4.850186e+11,6.746945e+08,-1.003920,-0.970297,34.867411,False
54,oracle_long_only_lambda_1000,oracle_nontradable,7.101169e+10,9.878220e+07,-1.256280,-0.970297,35.123129,False


In [3]:
predictability = pd.read_csv(card / 'predictability_summary.csv')
display(predictability.sort_values(['stable_signal', 'feature_set', 'target'], ascending=[False, True, True]))
outer = pd.read_csv(card / 'outer_year_risk_metrics.csv')
display(outer.groupby(['strategy_kind', 'cost_bps'])['risk_gate_pass'].agg(['sum', 'count']))

,feature_set,target,metric_name,mean_fold_metric,median_fold_metric,positive_years,fold_years,bootstrap_p05,bootstrap_p50,bootstrap_p95,stable_signal,selection_uses_outer_test
1,base,cash_advantage_1d,spearman_ic,0.088149,0.086115,7,7,0.006316,0.040041,0.074062,True,False
4,base,future_vol_5d,spearman_ic,0.353364,0.241779,7,7,0.508033,0.590157,0.655504,True,False
11,base_plus_soxs,cash_advantage_1d,spearman_ic,0.096115,0.092336,7,7,0.014434,0.048162,0.085607,True,False
13,base_plus_soxs,cash_advantage_5d,spearman_ic,0.143256,0.138113,7,7,0.005304,0.052947,0.120040,True,False
14,base_plus_soxs,future_vol_5d,spearman_ic,0.344545,0.246792,7,7,0.480323,0.569880,0.647935,True,False
0,base,cash_advantage_10d,spearman_ic,0.200915,0.180806,7,7,-0.056856,0.047310,0.158756,False,False
2,base,cash_advantage_20d,spearman_ic,0.262654,0.223877,7,7,-0.100825,0.040578,0.182123,False,False
3,base,cash_advantage_5d,spearman_ic,0.141012,0.148509,7,7,-0.022623,0.047655,0.116096,False,False
5,base,leverage_advantage_10d,spearman_ic,0.138489,0.098486,6,7,-0.085327,0.008609,0.111418,False,False
6,base,leverage_advantage_1d,spearman_ic,0.069609,0.081441,7,7,-0.003182,0.030997,0.068227,False,False


sum  count
strategy_kind       cost_bps            
mechanical_tradable 10         43     76
                    25         42     76
oracle_nontradable  10         34     40
                    25         34     40

## 最终中文结论

- 做了什么：用 qlib DatasetH/DataHandlerLP 和 recorder 跑固定机械规则、不可交易 oracle，以及 2016–2022 逐年 walk-forward 标签审计。
- 关键指标：以执行后生成的 `feasibility_frontier.csv`、`predictability_summary.csv` 和 recorder 大表为准。
- 区间差异：2011–2022 是开发证据；2023–2026 是已观察 outer，仅报告、不选择。
- 晋升决定：本卡只审计，不晋升模型。只有联合门槛可行且目标跨年稳定，才进入冻结 allocation model 和全新 sealed-forward。
- 风险：oracle 使用未来收益；现金收益设为零；SOXS 每日重置且有路径损耗；现有特征未必能预测 SOXL 相对 SOXX 的加杠杆优势。
- 下一步：依据审计结论只保留可预测目标，避免继续堆叠资产或阈值。